# Load dataset

In [2]:
import torch
from PIL import Image

# --- CONFIGURATION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
from ift6289_rag.dataset import load_data_vidore

subset = "physics"
lang = "english"
ds_corpus, ds_queries, ds_qrels, ds_metadata = load_data_vidore(subset, lang)



In [4]:
num_queries = len(ds_queries)
num_corpus = len(ds_corpus)
num_qrels = len(ds_qrels)
num_metadata = len(ds_metadata)

print(f"Number of queries: {num_queries}")
print(f"Number of corpus: {num_corpus}")


Number of queries: 302
Number of corpus: 1674


# Encode images

## Load model

In [9]:
from ift6289_rag.model import load_nemotron_colembed_model
model = load_nemotron_colembed_model()


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 916/916 [00:24<00:00, 36.74it/s, Materializing param=vision_model.vision_model.post_layernorm.weight]                       


## Pre-compute and save image embeddings

In [36]:
from ift6289_rag.features import precompute_image_embeddings
save_dir = "colembed_cache_pages_fp16"
precompute_image_embeddings(model, ds_corpus, save_dir=save_dir)

100%|██████████| 53/53 [00:54<00:00,  1.02s/it]


# Visual Retrieval

**If NDCG stays far below the paper (~47):** The cache may have been built when corpus order differed. Clear the cache and re-run `precompute_image_embeddings` so files `{corpus_id}.pt` match the current `ds_corpus`. Then re-run the "Load all the pages embeddings" cell and the NDCG evaluation.

In [7]:
from ift6289_rag.features import load_precomputed_image_embeddings
save_dir = "colembed_cache_pages_fp16"
pages_embeddings = load_precomputed_image_embeddings(ds_corpus, save_dir)


  0%|          | 0/1674 [00:00<?, ?it/s]

100%|██████████| 1674/1674 [01:16<00:00, 21.92it/s]


In [ ]:
from ift6289_rag.utils import evaluate_ndcg
ndcg_at_k = evaluate_ndcg(model, ds_queries, ds_qrels, ds_corpus, pages_embeddings=pages_embeddings, k=10)

NDCG@10:   0%|          | 0/302 [00:00<?, ?it/s]/home/fidaa/.cache/huggingface/modules/transformers_modules/nvidia/llama_hyphen_nemotron_hyphen_colembed_hyphen_vl_hyphen_3b_hyphen_v2/680b47b199f99bc0ec2f4e90ffa583ec0c2e452c/modeling_llama_nemotron_vl.py:173: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_bidirectional_mask`. Use `inputs_embeds` instead.
  return create_bidirectional_mask(
NDCG@10:  11%|█▏        | 34/302 [03:04<24:13,  5.42s/it]